# Решение: ЛР 01.3 — Сравнение моделей до/после отбора признаков

Ориентир по времени: 60 минут (+ опциональное расширение).

## Как читать это решение

- Главный фокус: сравнить модели на одинаковых feature sets и оценить trade-off качество/время.
- Сначала формируется `model_results`, затем агрегированный `summary`.
- После этого можно переходить к самостоятельным заданиям в TODO-ноутбуке.

## Цель
Сравнить качество и скорость классических моделей на полном наборе признаков и на наборах, полученных после отбора.

In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

ROOT = Path('..') if (Path('..') / 'lab_utils.py').exists() else Path('.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from lab_utils import (
    load_dataset,
    split_xy,
    train_test_split_stratified,
    build_preprocessor,
    transform_with_names,
    to_dense,
    evaluate_binary_model,
    metrics_to_long_rows,
)

SEED = 42
DATASETS = {
    'medical': ROOT / 'data' / 'medical_cardiovascular_risk.csv',
    'finance': ROOT / 'data' / 'finance_credit_risk.csv',
}
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

## Загрузка candidate feature set
Наборы берутся из `feature_sets_wrapper_embedded.json` (Notebook 2).

In [ ]:
feature_sets_path = OUTPUT_DIR / 'feature_sets_wrapper_embedded.json'
if feature_sets_path.exists():
    with open(feature_sets_path, 'r', encoding='utf-8') as f:
        feature_sets = json.load(f)
else:
    feature_sets = {}

print('feature sets loaded:', bool(feature_sets))

## Сравнение моделей
Обязательные модели:
- `LogisticRegression`
- `RandomForestClassifier`
- `LinearSVC`

Опциональный блок:
- `MLPClassifier` (один скрытый слой)

In [ ]:
# Учебный ориентир: цикл ниже строит полностью воспроизводимый эксперимент
# и сохраняет результаты в формате, удобном для сравнения между датасетами.
RUN_MLP_BLOCK = False

model_results_rows = []

for dataset_name, path in DATASETS.items():
    df = load_dataset(str(path))
    X, y = split_xy(df)
    X_train, X_test, y_train, y_test = train_test_split_stratified(X, y, random_state=SEED)

    preprocessor = build_preprocessor(X_train)
    X_train_t, X_test_t, feature_names = transform_with_names(preprocessor, X_train, X_test)
    X_train_t = to_dense(X_train_t)
    X_test_t = to_dense(X_test_t)
    feature_names = np.array(feature_names)

    available_sets = {'full': feature_names.tolist()}

    from_json = feature_sets.get(dataset_name, {})
    for set_name, feats in from_json.items():
        feats_filtered = [f for f in feats if f in set(feature_names)]
        if len(feats_filtered) >= 2:
            available_sets[set_name] = feats_filtered

    models = {
        'LogisticRegression': LogisticRegression(
            max_iter=2500,
            class_weight='balanced',
            random_state=SEED,
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=350,
            random_state=SEED,
            class_weight='balanced_subsample',
            n_jobs=-1,
        ),
        'LinearSVC': LinearSVC(
            C=1.0,
            class_weight='balanced',
            random_state=SEED,
        ),
    }

    if RUN_MLP_BLOCK:
        models['MLPClassifier'] = MLPClassifier(
            hidden_layer_sizes=(32,),
            max_iter=400,
            random_state=SEED,
            early_stopping=True,
        )

    for feature_set_name, selected_features in available_sets.items():
        idx = [int(np.where(feature_names == f)[0][0]) for f in selected_features]
        X_train_sel = X_train_t[:, idx]
        X_test_sel = X_test_t[:, idx]

        for model_name, model in models.items():
            metrics = evaluate_binary_model(model, X_train_sel, y_train, X_test_sel, y_test)
            model_results_rows.extend(
                metrics_to_long_rows(
                    dataset_name=dataset_name,
                    feature_set=feature_set_name,
                    model_name=model_name,
                    metrics=metrics,
                )
            )

model_results = pd.DataFrame(model_results_rows)
model_results.head(15)

## Итоги и экспорт `model_results`

In [ ]:
model_results_path = OUTPUT_DIR / 'model_results.csv'
model_results.to_csv(model_results_path, index=False)
print(f'Saved: {model_results_path}')

summary = (
    model_results
    .pivot_table(
        index=['dataset', 'feature_set', 'model'],
        columns='metric',
        values='value',
        aggfunc='mean',
    )
    .reset_index()
    .sort_values(['dataset', 'roc_auc', 'f1', 'accuracy'], ascending=[True, False, False, False])
)
summary.head(20)

## Контрольные точки
1. Убедитесь, что есть минимум метрики `accuracy`, `f1`, `roc_auc`.
2. Проверьте, что в `model_results` присутствуют оба датасета.
3. Сравните хотя бы `full` и один отобранный набор признаков.

In [ ]:
required_columns = {'dataset', 'feature_set', 'model', 'metric', 'value', 'fit_time_sec'}
assert required_columns.issubset(model_results.columns)

assert set(model_results['dataset']) == {'medical', 'finance'}
assert {'accuracy', 'f1', 'roc_auc'}.issubset(set(model_results['metric']))

for dataset_name in ['medical', 'finance']:
    subset = model_results[model_results['dataset'] == dataset_name]
    assert 'full' in set(subset['feature_set'])

print('Проверки пройдены успешно.')

## Типичные ошибки
- Несогласованные наборы признаков между train и test.
- Сравнение моделей только по одной метрике без учета времени обучения.
- Включение MLP без фиксации random_state и без проверки сходимости.

## Финальные выводы (пример решения)
1. Feature selection часто дает сопоставимое качество при меньшем числе признаков и лучшем времени обучения.
2. Универсального лучшего метода нет: выбор зависит от датасета и модели.
3. Для production-подхода разумно оставлять компактный набор признаков, если метрики не ухудшаются статистически значимо.